In [1]:
import numpy as np 
import torch 
import matplotlib.pyplot as plt 

In [5]:
# y = 3x + 2 + noise 
np.random.seed(42) 
X = np.random.randn(100, 1) 
y = 3 * X + 2 + 0.5 * np.random.randn(100, 1) 

# closed form ols: beta = (X'X)^-1 X'y 
# add biuas column 
X_b = np.hstack([np.ones_like(X), X]) 
beta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y 
print(f"Closed-form intercept = {beta[0, 0]:.4f}, slope={beta[1, 0]:.4f}")  

Closed-form intercept = 2.0037, slope=2.9284


In [ ]:
X_t = torch.tensor(X, dtype=torch.float32) 
y_t = torch.tensor(y, dtype=torch.float32) 

# grad tracking 
w = torch.randn(1, 1, requires_grad=True)
b = torch.randn(1, requires_grad=True)


lr = 0.05 
for epoch in range(200): 
    # literally slope y = mx + b   
    y_pred = X_t @ w + b 
    # squared error 
    loss = ((y_pred - y_t) ** 2).mean() 
    # just populates the graph 
    # each tensor has an attribute grad that is updated during graph traversal when loss.backward is called
    loss.backward()

    # update 
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
    if epoch % 50 == 0:
        print(f"epoch {epoch}: loss={loss.item():.4f}, w={w.item():.4f}, b={b.item():.4f}")

print(f"\nFinal: w={w.item():.4f}, b={b.item():.4f}")


epoch 0: loss=7.1295, w=0.7067, b=3.0442
epoch 50: loss=0.2217, w=2.9005, b=1.9988
epoch 100: loss=0.2209, w=2.9279, b=2.0035
epoch 150: loss=0.2209, w=2.9284, b=2.0037

Final: w=2.9284, b=2.0037


In [ ]:
# nn module 
import torch.nn as nn 
import torch.optim as optim 
# sets requires_grad=True 
model = nn.Linear(1, 1) # 1 input feature, 1 output 

loss_fn = nn.MSELoss() # mean square error 
optimizer = optim.SGD(model.parameters(), lr=0.05) # stochastic gradient descent 

for epoch in range(200): 
    # forward pass here  
    # pytorch invokes the __call__ method 
    # roughyl equivalent to model.forward(X_t)
    y_pred = model(X_t) 
    loss = loss_fn(y_pred, y_t) 

    optimizer.zero_grad() 
    loss.backward() 
    optimizer.step() 

    if epoch % 50 == 0: 
        print(f"epoch {epoch}: loss={loss.item():.4f}")
w_learned = model.weight.item() 
b_learned = model.bias.item() 
print(f"Final: w={w_learned:.4f}, b={b_learned:.4f}")

epoch 0: loss=9.7725
epoch 50: loss=0.2238
epoch 100: loss=0.2209
epoch 150: loss=0.2209
Final: w=2.9284, b=2.0037


1. training loop rhythm
```zero_grad -> forward -> loss -> backward -> step``` 

2. `requires_grad=True`
autograd switch. never set by hand. can only opt out. 

3. `nn.Parameter` - bridge. tensor wrapped in `nn.Parmeter` and assigned as a module attribute gets 2 things: `requires_grad=True` and automatic registration with `model.parameters()` (this is how optimizer finds weights)

4. `loss.backward()` side effect - returns None, walks the graphs, deposits gradients on grad during graph traversal when `requires_grad=True` 

5. gradients accumulate, they don't overwrite. never forget `optimizer.zero_grad()`. gradients sum across `.backward()` calls until you explicitly zero them. forgetting to zero the gradients results in successful training but learns garbage due to exploding gradients. 

6. `torch.no_grad()` disables tracking temporarily 

7. always call the module and never `.forward()`. use `model(X_t)`. the `__call__` wrapper hooks and handles training/eval state. `.forward()` skips that machinery 

8. inside a custom `forward`, the same rule applies recursively. `self.fc1(x)` not `self.fc1.forward(x)` 

9. `model.parameters()` is what the optimizer eats. this is how `optim.SGD(model.parameters(), ...)` knows what to update 

10. track the shape 

TLDR
PyTorch tracks gradients by default. Every iteration: zero, forward, loss, backward, step. The optimizer manages parameters; the module's __call__ runs the forward pass; .backward() populates .grad as a side effect.